# FastAPI SSE Streaming — Token-by-Token LangGraph Output
## Stream Tokens Over HTTP — FastAPI and Server-Sent Events
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/75-fastapi-sse-streaming/fastapi_sse_streaming_workbook.ipynb)

Streams LangGraph output token-by-token using astream_events and Server-Sent Events (SSE). The notebook demos the streaming graph in Colab; the FastAPI server component is shown as reference code.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — SSE vs WebSocket; streaming=True; astream_events protocol |
| 2 | **Streaming Graph** — think → answer nodes; streaming=True in ChatOpenAI |
| 3 | **astream_events()** — on_chat_model_stream events for token-level output |
| 4 | **FastAPI Reference** — EventSourceResponse pattern (reference — not run in Colab) |
| 5 | **Full Demo** — Live token streaming in Colab cell |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
import asyncio
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

class ThinkAnswerState(TypedDict):
    question: str
    answer: str

llm_stream = ChatOpenAI(model="gpt-4o-mini", streaming=True)
llm_plain  = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def think_node(state):
    r = llm_plain.invoke([SystemMessage(content="Think briefly about the question."), HumanMessage(content=state["question"])])
    return {"answer": r.content}

def answer_node(state):
    r = llm_plain.invoke([SystemMessage(content="Give a concise final answer."), HumanMessage(content=state["question"])])
    return {"answer": r.content}

g = StateGraph(ThinkAnswerState)
g.add_node("think_node", think_node)
g.add_node("answer_node", answer_node)
g.add_edge(START, "think_node"); g.add_edge("think_node", "answer_node"); g.add_edge("answer_node", END)
app = g.compile()

DEMO_QUESTION = "Explain how photosynthesis works in 2 sentences."

print("=== Standard invoke (no streaming) ===")
r = app.invoke({"question": DEMO_QUESTION, "answer": ""})
print(f"Answer: {r['answer']}")

print("\n=== Token-by-token streaming with astream_events ===")
async def stream_tokens():
    tokens = []
    async for event in app.astream_events({"question": DEMO_QUESTION, "answer": ""}, version="v1"):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                tokens.append(chunk)
                print(chunk, end="", flush=True)
    print()
    return tokens

tokens = asyncio.run(stream_tokens())
print(f"\nTotal tokens streamed: {len(tokens)}")

print("\n=== FastAPI SSE pattern (reference — requires running server) ===")
print("""
from fastapi import FastAPI
from sse_starlette.sse import EventSourceResponse

app_server = FastAPI()

@app_server.get("/stream")
async def stream_endpoint(question: str):
    async def generate():
        async for event in app.astream_events({"question": question, "answer": ""}, version="v1"):
            if event["event"] == "on_chat_model_stream":
                chunk = event["data"]["chunk"].content
                if chunk:
                    yield {"data": chunk}
    return EventSourceResponse(generate())
# Run with: uvicorn main:app_server --host 0.0.0.0 --port 8000
""")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*